# 10. Accessors Overview — 18개 accessor 한눈에

hwpapi 에는 **18개 accessor** 가 도메인별로 그룹핑되어 있습니다.

## 🚀 TL;DR — Discovery

```python
app = App()
app.help()               # 18개 accessor + context manager 출력
repr(app)                # App(visible=True, version='13.0.0', docs=2, page=5/20)
```

실제 `app.help()` 출력 모습:


```
App(visible=False, version='12, 0, 0, 3146', docs=1, page=1/1)

════════════════════════════════════════════════════════════════════
 hwpapi · App 사용 가능 API
════════════════════════════════════════════════════════════════════

· Navigation & Selection
    app.move              — 커서 이동 (.doc.top(), .line.end() ...)
    app.sel               — 현재 선택 제어 (.current_paragraph(), .compress())

· Collections
    app.documents         — 열린 문서 컬렉션
    app.fields            — 누름틀(필드) — dict/list-like
    app.bookmarks         — 책갈피 — .add(), .goto(), in
    app.hyperlinks        — 하이퍼링크 — .add(text, url)
    app.images            — 이미지 컬렉션 — .resize_all(), .grayscale_all()
    app.styles            — 문단 스타일 — .apply(name)
    app.controls          — 문서 내 모든 control 순회

· Structure
    app.cell              — 표 셀 조작
    app.table             — 표 구조 + 일괄 서식 — .header_row(), .align()
    app.page              — 페이지 속성

· Transform & View
    app.convert           — 숫자→한글, 폰트 교체, 줄 나눔
    app.view              — zoom, 전체화면, 페이지 모드

· Quality & Templates
    app.lint              — 문서 품질 체크 (callable → LintReport)
    app.template          — 템플릿 .save() / .apply()
    app.config            — App 기본 선호도

· Presets & Debug
    app.preset            — 꾸미기 프리셋 (title_box, striped_rows, toc …)
    app.debug             — .state(), .trace(), .timing()

· Context managers (with 구문으로 사용)
    with app.silenced(mode='yes')           — 다이얼로그 자동 응답 (종료 시 복원)
    with app.suppress_errors()              — 에러 dialog ABORT + Python 예외 swallow
    with app.batch_mode(hide=True)          — 창 숨김 + dialog 억제 — 대량 처리 5~10배 가속
    with app.undo_group(description)        — 블록 내 편집을 단일 undo 경계로
    with app.charshape_scope(**fmt)         — 블록 내 문자 모양 임시 변경
    with app.parashape_scope(**fmt)         — 블록 내 문단 모양 임시 변경
    with app.use_document(doc)              — 블록 내 활성 문서 일시 전환
    with app.debug.trace(verbose=True)      — 블록 내 COM Run() 호출 로그

· 주요 property
    app.text
    app.visible
    app.version
    app.page_count
    app.current_page
    app.selection
    app.charshape
    app.parashape
    app.status

' app.help()' 로 언제든 다시 확인할 수 있습니다.
════════════════════════════════════════════════════════════════════
```


In [ ]:
# app.help() 를 문서에 삽입
import io, contextlib
from hwpapi.core import App
buf = io.StringIO()
a = App(is_visible=False)
with contextlib.redirect_stdout(buf):
    a.help()
output = buf.getvalue()
# HWP 앱 종료
try: a.quit()
except Exception: pass
print(output)

## 18개 accessor 매트릭스

| 카테고리 | Accessor | 용도 |
|:---|:---|:---|
| **Navigation & Selection** | `app.move` | 커서 이동 |
| | `app.sel` | 선택 제어 |
| **Collections** | `app.documents` | 열린 문서 |
| | `app.fields` | 누름틀 |
| | `app.bookmarks` | 책갈피 |
| | `app.hyperlinks` | 하이퍼링크 |
| | `app.images` | 이미지 |
| | `app.styles` | 문단 스타일 |
| | `app.controls` | 모든 control |
| **Structure** | `app.cell` | 표 셀 |
| | `app.table` | 표 구조 + 배치 서식 |
| | `app.page` | 페이지 설정 |
| **Transform & View** | `app.convert` | 숫자/폰트/줄 나눔 |
| | `app.view` | zoom, 모드 |
| **Quality & Templates** | `app.lint` | 문서 품질 체크 |
| | `app.template` | 템플릿 save/apply |
| | `app.config` | App 선호도 |
| **Presets & Debug** | `app.preset` | 꾸미기 프리셋 11종 |
| | `app.debug` | 디버깅 도구 |


## 주요 예제

### 1. Fields — dict-style mail merge

```python
app.fields["name"] = "홍길동"
app.fields.update({"date": "2026-04-15", "amount": "1,200,000"})
print(app.fields.to_dict())
# {'name': '홍길동', 'date': '2026-04-15', 'amount': '1,200,000'}
```

실제 렌더링 결과:


![demo — fields_dict_full](img/demo_fields_dict_full.png)


*위: `{{tag}}` → 필드 자동 변환 → dict-style 일괄 값 주입 결과*

### 2. Table batch formatting

```python
app.insert_table(rows=3, cols=4)
app.table.header_row(color="sky", text_color="#FFFFFF")
app.table.current_row(bg="#FFFFCC")
app.table.align(horz="right", scope="current_col")
```

실제 렌더링 결과:


![demo — table_batch_ops](img/demo_table_batch_ops.png)


*위: `table.header_row()` 스카이블루 제목행 적용 결과*

### 3. Selection + compress/expand

```python
app.sel.current_paragraph()
app.sel.compress(step=2)       # 자간·장평 동시 축소
```


![demo — sel_operations](img/demo_sel_operations.png)


*위: `sel.current_paragraph()` 후 서식 적용 결과 (한 문단만 강조)*

### 4. Context manager 사용

```python
with app.batch_mode():                       # 5~10배 가속
    for row in df.iterrows():
        app.fields.update(row.to_dict())
        app.save(f"out/{row['name']}.hwp")

with app.charshape_scope(bold=True, text_color="#FF0000"):
    app.insert_text("강조 텍스트")
# 블록 종료 시 이전 charshape 로 자동 복원
```

실제 `charshape_scope` 동작:


![demo — charshape_scope_real](img/demo_charshape_scope_real.png)


*위: `charshape_scope` 블록 내부만 굵은 빨간색 큰 글씨, 블록 종료 시 자동 복원*

### 5. Convert — 숫자 → 한글숫자

```python
>>> app.convert.number_to_korean("1,234,567")
'백이십삼만사천오백육십칠'

>>> app.convert.number_to_korean("10,000")
'일만'

>>> app.convert.number_to_korean("100,000,000")
'일억'
```

실제 실행 결과:


```
정수 → 한글숫자
        1,234,567 → 백이십삼만사천오백육십칠
           10,000 → 일만
      100,000,000 → 일억
              365 → 삼백육십오
        5,000,000 → 오백만
```


### 6. Fluent — 체이닝

```python
(app
 .insert_heading("제목", level=1)
 .insert_text("본문 첫 줄")
 .insert_paragraph_break()
 .insert_text("본문 두 번째 줄")
 .insert_table(rows=3, cols=2)
 .save("report.hwp"))
```

실제 체이닝으로 생성한 문서:


![demo — fluent_chain](img/demo_fluent_chain.png)


*위: Fluent API 체이닝으로 한 번에 작성한 문서*

### 7. View — zoom / modes

```python
app.view.zoom(150)              # 확대 150%
app.view.zoom_fit_page()        # 페이지 맞춤
app.view.page_mode()            # 일반 편집 모드
app.view.draft_mode()           # 초안 모드 (빠른 렌더)
app.view.full_screen()          # 전체 화면
app.view.scroll_to_cursor()
```

> View 제어는 HWP 화면 설정이라 PDF 출력에는 영향이 없습니다.
> 실제 HWP GUI 에서만 변화를 확인할 수 있습니다.


## 마치며

각 accessor 의 자세한 사용법과 시각 예제:
- 프리셋 → [11_presets_gallery](11_presets_gallery.ipynb)
- 대량 처리 / workflow → [12_batch_and_workflow](12_batch_and_workflow.ipynb)
- 디버깅 / 품질 / 설정 → [13_debugging_tools](13_debugging_tools.ipynb)
- 레거시 → 신 API 마이그레이션 표 → [docs/API_GUIDE.md](../../docs/API_GUIDE.md)
